In [5]:
import sympy as sp
from IPython.display import display, Math

def resolver_edo_fourier_separado(coef_edo, f_t, variable_t, periodo_T, inicio_intervalo, N=3):
    t = variable_t
    a, b, c = coef_edo
    w0 = 2 * sp.pi / periodo_T
    fin = inicio_intervalo + periodo_T
    n = sp.Symbol('n', integer=True)
    C1, C2 = sp.symbols('A B')

    # 0. SOLUCIÓN HOMOGÉNEA y_H(t)
    y_func = sp.Function('y')
    eq_hom = sp.Eq(a*y_func(t).diff(t, 2) + b*y_func(t).diff(t, 1) + c*y_func(t), 0)
    sol_hom = sp.dsolve(eq_hom, y_func(t))
    
    if a != 0:
        y_H = sol_hom.rhs.subs([(sp.Symbol('C1'), C1), (sp.Symbol('C2'), C2)])
    else:
        y_H = sol_hom.rhs.subs(sp.Symbol('C1'), C1)

    print("=" * 80)
    print("0. SOLUCIÓN HOMOGÉNEA DE LA EDO (y_H)")
    print("=" * 80)
    display(Math(rf"y_H(t) = {sp.latex(y_H)}"))

    # =========================================================================
    # 1. MÉTODO: SERIE TRIGONOMÉTRICA DE FOURIER (STF)
    # =========================================================================
    print("\n" + "=" * 80)
    print("1. MÉTODO POR SERIE TRIGONOMÉTRICA DE FOURIER (STF)")
    print("=" * 80)

    # Coeficientes STF
    a0 = sp.simplify((2 / periodo_T) * sp.integrate(f_t, (t, inicio_intervalo, fin)))
    an_expr = sp.simplify((2 / periodo_T) * sp.integrate(f_t * sp.cos(n * w0 * t), (t, inicio_intervalo, fin)))
    bn_expr = sp.simplify((2 / periodo_T) * sp.integrate(f_t * sp.sin(n * w0 * t), (t, inicio_intervalo, fin)))

    display(Math(rf"\text{{Coeficientes STF: }} a_0 = {sp.latex(a0)}, \quad a_n = {sp.latex(an_expr)}, \quad b_n = {sp.latex(bn_expr)}"))

    # Construcción de y_P(t) en STF
    y_p_stf = a0 / (2 * c) if c != 0 else sp.S(0)

    for k in range(1, N + 1):
        a_k = sp.simplify(an_expr.subs(n, k))
        b_k = sp.simplify(bn_expr.subs(n, k))
        if a_k.has(sp.nan, sp.zoo): a_k = sp.limit(an_expr, n, k)
        if b_k.has(sp.nan, sp.zoo): b_k = sp.limit(bn_expr, n, k)

        s_k = sp.I * k * w0
        P_s = a * s_k**2 + b * s_k + c

        if P_s != 0:
            alpha_k = c - a * (k * w0)**2
            beta_k = b * k * w0
            det = alpha_k**2 + beta_k**2
            A_k = sp.simplify((alpha_k * a_k - beta_k * b_k) / det)
            B_k = sp.simplify((beta_k * a_k + alpha_k * b_k) / det)
            y_p_stf += A_k * sp.cos(k * w0 * t) + B_k * sp.sin(k * w0 * t)
        else:
            A_k = sp.simplify(-b_k / (2 * a * k * w0)) if a != 0 else sp.simplify(a_k / b)
            B_k = sp.simplify(a_k / (2 * a * k * w0)) if a != 0 else sp.simplify(b_k / b)
            y_p_stf += t * (A_k * sp.cos(k * w0 * t) + B_k * sp.sin(k * w0 * t))

    y_gen_stf = y_H + y_p_stf

    display(Math(rf"\text{{Entrada }} f(t)_{{\text{{STF}}}} = \frac{{{sp.latex(a0)}}}{{2}} + \sum_{{n=1}}^{{\infty}} \left[ \left({sp.latex(an_expr)}\right)\cos(n\omega_0 t) + \left({sp.latex(bn_expr)}\right)\sin(n\omega_0 t) \right]"))
    display(Math(rf"\text{{Solución Particular }} y_{{P,\text{{STF}}}}(t) \approx {sp.latex(sp.simplify(y_p_stf))}"))
    display(Math(rf"\text{{Solución General }} y_{{\text{{STF}}}}(t) \approx {sp.latex(sp.simplify(y_gen_stf))}"))

    # =========================================================================
    # 2. MÉTODO: SERIE COMPLEJA DE FOURIER (SCF)
    # =========================================================================
    print("\n" + "=" * 80)
    print("2. MÉTODO POR SERIE COMPLEJA DE FOURIER (SCF)")
    print("=" * 80)

    # Coeficiente SCF
    cn_expr = sp.simplify((1 / periodo_T) * sp.integrate(f_t * sp.exp(-sp.I * n * w0 * t), (t, inicio_intervalo, fin)))
    display(Math(rf"\text{{Coeficiente SCF: }} c_n = {sp.latex(cn_expr)}"))

    # Construcción de y_P(t) en SCF
    y_p_scf = sp.S(0)

    for k in range(-N, N + 1):
        c_k = sp.simplify(cn_expr.subs(n, k))
        if c_k.has(sp.nan, sp.zoo): c_k = sp.limit(cn_expr, n, k)

        s_k = sp.I * k * w0
        P_s = a * s_k**2 + b * s_k + c

        if P_s != 0:
            C_k = sp.simplify(c_k / P_s)
            y_p_scf += C_k * sp.exp(sp.I * k * w0 * t)
        else:
            P_prime = 2 * a * s_k + b
            C_k = sp.simplify(c_k / P_prime)
            y_p_scf += C_k * t * sp.exp(sp.I * k * w0 * t)

    y_gen_scf = y_H + y_p_scf

    display(Math(rf"\text{{Entrada }} f(t)_{{\text{{SCF}}}} = \sum_{{n=-\infty}}^{{\infty}} \left({sp.latex(cn_expr)}\right) e^{{i n \omega_0 t}}"))
    display(Math(rf"\text{{Solución Particular (Forma Compleja) }} y_{{P,\text{{SCF}}}}(t) \approx {sp.latex(sp.simplify(y_p_scf))}"))
    display(Math(rf"\text{{Solución General (Forma Compleja) }} y_{{\text{{SCF}}}}(t) \approx {sp.latex(sp.simplify(y_gen_scf))}"))

    return y_gen_stf, y_gen_scf

# --- EJECUCIÓN CON TUS EJERCICIOS ---
t = sp.Symbol('t', real=True)
T = 2 * sp.pi
f_t = sp.Piecewise((1, (t >= 0) & (t < sp.pi)), (0, (t >= sp.pi) & (t <= 2 * sp.pi)))

# Para el Problema 1: y' + y = f(t) -> coef_edo = (0, 1, 1)
# Para el Problema 2: y'' + y = f(t) -> coef_edo = (1, 0, 1)
sol_stf, sol_scf = resolver_edo_fourier_separado((1, 0, 1), f_t, t, T, 0, N=3)

0. SOLUCIÓN HOMOGÉNEA DE LA EDO (y_H)


<IPython.core.display.Math object>


1. MÉTODO POR SERIE TRIGONOMÉTRICA DE FOURIER (STF)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

...


2. MÉTODO POR SERIE COMPLEJA DE FOURIER (SCF)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [1]:
import sympy as sp
from IPython.display import display, Math

def resolver_ejercicio_tarea(coef_edo, f_t, variable_t, periodo_T, inicio_intervalo, titulo=""):
    t = variable_t
    a, b, c = coef_edo
    w0 = 2 * sp.pi / periodo_T
    fin = inicio_intervalo + periodo_T
    
    n = sp.Symbol('n', integer=True, positive=True)
    An_sym, Bn_sym = sp.symbols('A_n B_n')
    
    print("=" * 85)
    print(f"  {titulo.upper()}")
    print("=" * 85)
    
    # -------------------------------------------------------------
    # PASO 1: EDO DEL SISTEMA Y SOLUCIÓN HOMOGÉNEA
    # -------------------------------------------------------------
    y_func = sp.Function('y')(t)
    edo_eq = a * y_func.diff(t, 2) + b * y_func.diff(t, 1) + c * y_func
    
    C1, C2 = sp.symbols('C_1 C_2')
    sol_hom = sp.dsolve(sp.Eq(edo_eq, 0), y_func).rhs
    y_H = sol_hom.subs([(sp.Symbol('C1'), C1), (sp.Symbol('C2'), C2)]) if a != 0 else sol_hom.subs(sp.Symbol('C1'), C1)
        
    print("\n>>> PASO 1: PLANTEAMIENTO DE LA EDO Y SOLUCIÓN HOMOGÉNEA y_H(t)")
    display(Math(rf"\text{{EDO: }} {sp.latex(edo_eq)} = f(t)"))
    display(Math(rf"\text{{Solución Homogénea: }} y_H(t) = {sp.latex(y_H)}"))

    # -------------------------------------------------------------
    # PASO 2: SERIE TRIGONOMÉTRICA Y COMPLEJA DE LA ENTRADA f(t)
    # -------------------------------------------------------------
    a0 = sp.simplify((2 / periodo_T) * sp.integrate(f_t, (t, inicio_intervalo, fin)))
    an_expr = sp.simplify((2 / periodo_T) * sp.integrate(f_t * sp.cos(n * w0 * t), (t, inicio_intervalo, fin)))
    bn_expr = sp.simplify((2 / periodo_T) * sp.integrate(f_t * sp.sin(n * w0 * t), (t, inicio_intervalo, fin)))
    
    n_comp = sp.Symbol('n', integer=True)
    cn_expr = sp.simplify((1 / periodo_T) * sp.integrate(f_t * sp.exp(-sp.I * n_comp * w0 * t), (t, inicio_intervalo, fin)))

    print("\n>>> PASO 2: SERIE DE FOURIER DE LA ENTRADA f(t)")
    display(Math(rf"a_0 = {sp.latex(a0)}, \quad a_n = {sp.latex(an_expr)}, \quad b_n = {sp.latex(bn_expr)}"))
    display(Math(rf"f(t)_{{\text{{STF}}}} = \frac{{{sp.latex(a0)}}}{{2}} + \sum_{{n=1}}^{{\infty}} \left[ \left({sp.latex(an_expr)}\right) \cos(n t) + \left({sp.latex(bn_expr)}\right) \sin(n t) \right]"))
    display(Math(rf"c_n = {sp.latex(cn_expr)} \quad \implies \quad f(t)_{{\text{{SCF}}}} = \sum_{{n=-\infty}}^{{\infty}} \left({sp.latex(cn_expr)}\right) e^{{i n t}}"))

    # -------------------------------------------------------------
    # PASO 3: PROPUESTA DE SOLUCIÓN PARTICULAR Y DERIVADAS
    # -------------------------------------------------------------
    print("\n>>> PASO 3: PROPUESTA DE SOLUCIÓN PARTICULAR Y DERIVADAS")
    display(Math(rf"y_p(t) = A_0 + \sum_{{n=1}}^{{\infty}} \left( A_n \cos(n t) + B_n \sin(n t) \right)"))
    if b != 0:
        display(Math(rf"y_p'(t) = \sum_{{n=1}}^{{\infty}} \left( -n A_n \sin(n t) + n B_n \cos(n t) \right)"))
    if a != 0:
        display(Math(rf"y_p''(t) = \sum_{{n=1}}^{{\infty}} \left( -n^2 A_n \cos(n t) - n^2 B_n \sin(n t) \right)"))

    # -------------------------------------------------------------
    # PASO 4: SISTEMA DE ECUACIONES PARA COEFICIENTES (STF)
    # -------------------------------------------------------------
    alpha = c - a * (n * w0)**2
    beta = b * n * w0
    
    eq_cos = sp.Eq(alpha * An_sym + beta * Bn_sym, an_expr)
    eq_sin = sp.Eq(-beta * An_sym + alpha * Bn_sym, bn_expr)
    
    sol_coef = sp.solve([eq_cos, eq_sin], (An_sym, Bn_sym))
    An_formula = sp.simplify(sol_coef[An_sym])
    Bn_formula = sp.simplify(sol_coef[Bn_sym])
    
    print("\n>>> PASO 4: IGUALANDO COEFICIENTES SEMEJANTES")
    display(Math(rf"\text{{Igualando Cosenos: }} {sp.latex(alpha * An_sym + beta * Bn_sym)} = {sp.latex(an_expr)}"))
    display(Math(rf"\text{{Igualando Senos: }} {sp.latex(-beta * An_sym + alpha * Bn_sym)} = {sp.latex(bn_expr)}"))
    display(Math(rf"A_n = {sp.latex(An_formula)}"))
    display(Math(rf"B_n = {sp.latex(Bn_formula)}"))

    # -------------------------------------------------------------
    # PASO 5: EVALUACIÓN ARMÓNICA (N=1 A 3) Y CASOS DE RESONANCIA
    # -------------------------------------------------------------
    print("\n>>> PASO 5: EVALUACIÓN DE ARMÓNICOS Y RESOLUCIÓN DE RESONANCIA")
    y_p_trig = a0 / (2 * c) if c != 0 else sp.S(0)
    
    for k in range(1, 4):
        a_k = sp.simplify(an_expr.subs(n, k))
        b_k = sp.simplify(bn_expr.subs(n, k))
        
        s_k = sp.I * k * w0
        P_s = a * s_k**2 + b * s_k + c
        
        if P_s != 0:
            A_k = sp.simplify(An_formula.subs(n, k))
            B_k = sp.simplify(Bn_formula.subs(n, k))
            term_k = A_k * sp.cos(k * w0 * t) + B_k * sp.sin(k * w0 * t)
            display(Math(rf"n = {k}: \quad A_{k} = {sp.latex(A_k)}, \quad B_{k} = {sp.latex(B_k)} \quad \implies \quad y_{{p,{k}}}(t) = {sp.latex(term_k)}"))
        else:
            # Caso de resonancia (multiplicación por t)
            A_k = sp.simplify(-b_k / (2 * a * k * w0)) if a != 0 else sp.simplify(a_k / b)
            B_k = sp.simplify(a_k / (2 * a * k * w0)) if a != 0 else sp.simplify(b_k / b)
            term_k = t * (A_k * sp.cos(k * w0 * t) + B_k * sp.sin(k * w0 * t))
            display(Math(rf"n = {k} \text{{ (\textbf{{RESONANCIA!}}): }} \quad y_{{p,{k}}}(t) = {sp.latex(term_k)}"))
            
        y_p_trig += term_k

    # -------------------------------------------------------------
    # PASO 6: SOLUCIÓN COMPLETA (GENERAL = HOMOGÉNEA + PARTICULAR)
    # -------------------------------------------------------------
    y_gen_final = y_H + y_p_trig
    
    print("\n>>> PASO 6: SOLUCIÓN GENERAL Y PARTICULAR FINAL TRUNCADA (N=3)")
    display(Math(rf"y_p(t) \approx {sp.latex(sp.simplify(y_p_trig))}"))
    display(Math(rf"y(t) = y_H(t) + y_p(t) \approx {sp.latex(sp.simplify(y_gen_final))}"))
    print("\n" + "=" * 85 + "\n")


# =============================================================================
# EJECUCIÓN PARA LOS DOS PROBLEMAS DE LA TAREA
# =============================================================================
t = sp.Symbol('t', real=True)
T = 2 * sp.pi
f_t = sp.Piecewise(((0.25*t**2), (t >= 0) & (t < 2 * sp.pi)),)# (0, (t >= sp.pi) & (t <= 2 * sp.pi)))

# PROBLEMA 1: y' + y = f(t) -> coef_edo = (0, 1, 1)
resolver_ejercicio_tarea(
    coef_edo=(0, 1, 3),
    f_t=f_t,
    variable_t=t,
    periodo_T=T,
    inicio_intervalo=0,
    titulo="Problema 1: y' + y = f(t)"
)

# PROBLEMA 2: y'' + y = f(t) -> coef_edo = (1, 0, 1)
resolver_ejercicio_tarea(
    coef_edo=(1, 0, 1),
    f_t=f_t,
    variable_t=t,
    periodo_T=T,
    inicio_intervalo=0,
    titulo="Problema 2: y'' + y = f(t)"
)

  PROBLEMA 1: Y' + Y = F(T)

>>> PASO 1: PLANTEAMIENTO DE LA EDO Y SOLUCIÓN HOMOGÉNEA y_H(t)


<IPython.core.display.Math object>

<IPython.core.display.Math object>


>>> PASO 2: SERIE DE FOURIER DE LA ENTRADA f(t)


<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>


>>> PASO 6: SOLUCIÓN GENERAL Y PARTICULAR FINAL TRUNCADA (N=3)


<IPython.core.display.Math object>

<IPython.core.display.Math object>